# step08_artifacts_report — Reproducibility and Final Report

Record how to recreate the analysis and assemble a report draft grounded in the generated artifacts.

Run cells from top to bottom. Shared helpers and file paths are in `analysis_common.py`. Outputs go to `analysis_results/` and `figures/`.

In [1]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis' / 'analysis_common.py').exists() and ROOT.parent != ROOT: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'analysis'))
from analysis_common import *
# Inventory expected analysis artifacts and record reproducibility settings.
manifest=[]
for folder in [OUTPUT_DIR,FIGURES_DIR]:
    for p in sorted(folder.glob('*')):
        if p.is_file(): manifest.append({'path':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'modified_utc':pd.Timestamp(p.stat().st_mtime,unit='s',tz='UTC')})
manifest=pd.DataFrame(manifest); save_table(manifest,'step08_artifact_manifest.csv'); display(manifest)
requirements='''Python 3.10+\npandas\nnumpy\nmatplotlib\nrequests (Step 5 diff retrieval)\nJupyter Notebook or JupyterLab\n'''
(OUTPUT_DIR/'requirements-analysis.txt').write_text(requirements,encoding='utf-8')


,path,bytes,modified_utc
0,analysis\analysis_results\analysis_dataset.csv,76561399,2026-11-10 16:19:15.921265125+00:00
1,analysis\analysis_results\annual_article_activ...,20880,2026-11-10 16:18:51.411809206+00:00
2,analysis\analysis_results\article_comparison_m...,6066,2026-11-10 16:19:15.929359674+00:00
3,analysis\analysis_results\article_quarterly_ac...,138798,2026-11-10 16:10:35.847456694+00:00
4,analysis\analysis_results\article_revision_cov...,1952,2026-11-10 16:10:17.473204374+00:00
...,...,...,...
69,analysis\figures\step04_contributor_concentrat...,212290,2026-11-10 16:58:47.614394903+00:00
70,analysis\figures\step04_editor_types.png,69987,2026-11-10 16:58:46.269867182+00:00
71,analysis\figures\step04_minor_vs_activity.png,48669,2026-11-10 16:58:48.160663128+00:00
72,analysis\figures\step06_category_activity.png,81211,2026-11-10 16:59:26.820922613+00:00


101

## Reproduction instructions

1. Keep `historical_analysis/cleaned/historical_revisions_cleaned.csv` in place.
2. Install the packages listed in `analysis_results/requirements-analysis.txt`.
3. Open this project directory as the working directory.
4. Run `step01_dataset_validation.ipynb` through `step08_artifacts_report.ipynb` in order.
5. Review retrieval failures in Step 5 and add source citations to the Step 2 context review before making causal claims.
6. Generated tables are written to `analysis/analysis_results`; figures are written to `analysis/figures`.

The original plan requests a 6–8 page report and team contributions. Those require human-provided repository link and team-member contribution details, so this notebook creates a report draft with explicit fill-in fields instead of inventing them.

In [2]:
# Draft report text from the plan's required headings and recorded findings.
summary=pd.read_csv(OUTPUT_DIR/'step01_dataset_summary.csv').iloc[0]
findings=(OUTPUT_DIR/'step07_findings_summary.md').read_text(encoding='utf-8')
report=f'''# Wikipedia Digital Archaeology Project — Report Draft\n\n## Objective and Motivation\nThis project studies how Wikipedia articles documenting major historical events evolve, when editing activity changes, how participation is distributed, and what revision patterns can reveal.\n\n## Data and Data Collection\nThe analysis uses the cleaned revision metadata at `historical_analysis/cleaned/historical_revisions_cleaned.csv`. It contains {int(summary.revision_records):,} revision records across {int(summary.articles)} articles. Collection details should be checked against the project collection script before submission.\n\n## Data Processing and Quality\nTimestamps are parsed as UTC, numeric identifiers and article sizes are normalized, and revisions are ordered within article by timestamp and revision ID. See `step01_validation_report.csv` and `step01_field_completeness.csv` for observed quality metrics.\n\n## Methods or Implementation\nThe notebooks calculate per-period activity, robust monthly outliers (median + 3 × 1.4826 × MAD per article), first-to-last observed article-size change, contributor participation and concentration, and category summaries using documented primary categories. Revision diffs are retrieved only for a selected sample.\n\n## Results\n{findings}\n\n## Problems, Limitations, and Interpretation\nThe dataset is metadata rather than the full text of every revision. Anomaly detection identifies candidate periods but cannot explain their causes. Article byte counts include formatting, references, and other non-prose. Usernames and user IDs do not establish identity, expertise, or reliability. Category assignments are researcher-defined and results are sensitive to the mapping and article mix.\n\n## Future Work\nComplete manual review of retrieved diffs, investigate selected spikes with independent dated sources, verify the collection process and sampling, and refine the category mapping with project collaborators.\n\n## Team Contributions\n[Add each team member and their verified contribution.]\n\n## Repository\n[Insert the final repository link.]\n'''
(OUTPUT_DIR/'final_report_draft.md').write_text(report,encoding='utf-8'); print('Wrote',OUTPUT_DIR/'final_report_draft.md')


Wrote C:\Users\Jackson\Desktop\class\arch\finalProject\analysis\analysis_results\final_report_draft.md


In [3]:
# Expand the report draft with observed results, evidence tables, and explicit limits.
summary = pd.read_csv(OUTPUT_DIR/'step01_dataset_summary.csv').iloc[0]
coverage = pd.read_csv(OUTPUT_DIR/'step01_article_coverage.csv')
missing = pd.read_csv(OUTPUT_DIR/'step01_field_completeness.csv')
parents = pd.read_csv(OUTPUT_DIR/'step01_parent_link_validation.csv').iloc[0]
growth = pd.read_csv(OUTPUT_DIR/'step03_article_growth.csv')
editor_types = pd.read_csv(OUTPUT_DIR/'step04_editor_type_summary.csv')
editor_metrics = pd.read_csv(OUTPUT_DIR/'step04_article_editor_metrics.csv')
category_summary = pd.read_csv(OUTPUT_DIR/'step06_category_summary.csv')
spikes = pd.read_csv(OUTPUT_DIR/'step02_article_monthly_spikes.csv')
spikes = spikes[spikes.is_spike].sort_values('revisions', ascending=False)
context = pd.read_csv(OUTPUT_DIR/'step02_spike_context_review.csv')
diff_status = pd.read_csv(OUTPUT_DIR/'step05_revision_diffs_raw.csv')
case_studies = pd.read_csv(OUTPUT_DIR/'step05_revision_case_studies.csv')
second_revert_status = diff_status.loc[diff_status.revision_id.eq(30587594),'retrieval_status']
if not second_revert_status.empty and second_revert_status.iloc[0] == 'retrieved':
    second_revert_note = 'Revision 30587594 also has a retrieved diff that removes the repeated insertion about six seconds later.'
else:
    second_revert_note = 'Revision 30587594 has a revert edit summary and matching negative size change, but its diff still needs retrieval to verify the exact removal.'

def md_table(frame, columns, limit=8):
    view = frame[columns].head(limit).fillna('').astype(str)
    return '| ' + ' | '.join(columns) + ' |\n| ' + ' | '.join(['---']*len(columns)) + ' |\n' + '\n'.join('| ' + ' | '.join(v.replace('|','\\|') for v in row) + ' |' for row in view.to_numpy())

registered = editor_types.loc[editor_types.editor_type.eq('Registered (positive user ID)'), 'share_pct'].iloc[0]
anonymous = editor_types.loc[editor_types.editor_type.eq('Anonymous (user ID 0)'), 'share_pct'].iloc[0]
median_top10 = editor_metrics.top10_revision_share_pct.median()
largest_growth = growth.iloc[0]
context_note = 'No selected spike has an independently sourced contextual note in the review table.'
context_rows = context[context.source_url.notna() & context.source_url.ne('')]
if not context_rows.empty:
    row = context_rows.iloc[0]
    context_note = f"For {row.article_title} in {str(row.timestamp)[:7]}, an external historical timeline documents contemporaneous events. This supports temporal context only and does not show those events caused editing activity. Source: [{row.source_title}]({row.source_url})."

report = f'''# Wikipedia Digital Archaeology Project — Analysis Report Draft

## Objective and Motivation

This project examines how Wikipedia articles about selected historical events evolve in the collected revision record. It asks when editing activity changes, how article size changes, how contributor activity is distributed, and whether editing patterns differ across event categories. Revision counts and byte sizes are observable traces of editorial activity; on their own, they do not establish the accuracy, completeness, or historical importance of the resulting content.

## Data and Data Collection

The analysis uses `historical_analysis/cleaned/historical_revisions_cleaned.csv`, a cleaned metadata file with {int(summary.revision_records):,} revision records and {int(summary.articles)} articles. Valid timestamps span {summary.earliest_timestamp_utc} to {summary.latest_timestamp_utc}, approximately {summary.coverage_days/365.2425:.1f} years. The data includes article title, revision and parent IDs, timestamp, username and user ID, edit summary, article size, and a minor-edit flag. It does not include the full text of each revision. The project collection scripts and sampling settings should be checked before describing this file as a complete revision history.

The average article has {summary.mean_revisions_per_article:,.1f} collected records and the median has {summary.median_revisions_per_article:,.0f}. The largest record counts are concentrated in a few pages:

{md_table(coverage, ['article_title','revisions','share_of_records_pct'], 6)}

The five largest pages by record count contribute {coverage.head(5).share_of_records_pct.sum():.1f}% of the dataset. This uneven coverage matters when interpreting pooled activity: totals reflect both article attention and the choice of articles in the project.

## Data Processing and Quality

Timestamps were parsed as UTC, identifiers and sizes coerced to numeric values, blank usernames treated as unavailable, and records sorted by article, timestamp, and revision ID before calculating consecutive observed size changes. No duplicate revision IDs, missing revision IDs, missing timestamps, negative sizes, or out-of-order input rows were found in the validation pass. Edit summaries were blank for {int(missing.loc[missing.field.eq('edit_comment'),'missing_count'].iloc[0]):,} records ({missing.loc[missing.field.eq('edit_comment'),'missing_percent'].iloc[0]:.1f}%). Other required fields in the table below had no missing values.

Parent-link checks found {int(parents.parent_not_in_collected_dataset)} parent IDs absent from the collected dataset and {int(parents.parent_timestamp_not_earlier_than_child)} records whose parent timestamp is equal to or later than the child timestamp. These rows are preserved in `step01_parent_link_anomalies.csv` for inspection. Some absent parents may reflect collection boundaries, but the non-earlier timestamps warrant checking against the source history before making fine-grained temporal claims. The largest observed article sizes are also exported because a small number exceed 1 MB; size-based outliers were not silently removed.

## Methods

Monthly and annual revision counts were calculated by article and across the full collection. Monthly spike screening used an article-specific robust threshold: median plus three scaled median absolute deviations. Where the median absolute deviation is zero, the analysis falls back to the upper Tukey fence; when both measures indicate no variation, it flags no observations. This avoids labeling every above-median count as anomalous in sparse series. The resulting table is a screening tool, not a causal explanation.

Article growth compares the first and last observed sizes in the collected record. It reports absolute byte change and percentage change. Percentage change is especially sensitive when the first observed size is very small, so absolute change is presented alongside it. These are observed endpoints, not necessarily page creation and current size. Contributor type is proxied by user ID: zero is classified anonymous, positive values registered, and missing values unavailable. Top-contributor shares are based on usernames within each article; a username does not establish a person's identity. One primary historical-event category was assigned to each article and documented in `step06_article_category_mapping.csv`.

## Results

### Activity and editing spikes

The highest record count belongs to World War II ({coverage.iloc[0].revisions:,}), followed by September 11 attacks ({coverage.iloc[1].revisions:,}). That distinction is descriptive and does not mean these events are historically more important. The per-article month table identifies the following large candidate spikes:

{md_table(spikes, ['article_title','timestamp','revisions','article_median','robust_threshold'], 8)}

The Arab Spring page has a 2,050-record month in February 2011 against a per-month median of 16. An external 2011 timeline documents major contemporaneous Arab Spring developments, including Egypt's February transition. {context_note} Other spikes remain uninvestigated; the revision counts alone cannot identify anniversaries, media coverage, controversies, or other causes.

The record spans over 25 years, but the files do not provide article publication dates or an independently verified complete history for each page. Therefore, comparisons of first and last observed activity cannot determine whether interest peaks soon after an event or page publication. Further work should establish article creation dates and confirm collection coverage before making that comparison.

### Article size and growth

The largest net observed size increase is {largest_growth.article_title}: {largest_growth.net_change_bytes:,.0f} bytes, from {largest_growth.first_size_bytes:,.0f} to {largest_growth.last_size_bytes:,.0f} bytes. The next largest observed increases are summarized here:

{md_table(growth, ['article_title','first_size_bytes','last_size_bytes','net_change_bytes','growth_pct'], 6)}

Percentage gains for pages whose earliest collected size was only a few dozen or a few hundred bytes become extremely large. Those percentages should not be compared as if they measured substantive historical knowledge. The largest consecutive size changes in Step 3 include extreme values; their revision IDs are preserved in the candidate table, but a byte delta cannot say what material was added or removed.

### Contributor participation

Across all records, {registered:.1f}% have a positive user ID, {anonymous:.1f}% have user ID zero, and the small remainder have unavailable IDs. The median share of revisions attributed to the ten most frequent usernames is {median_top10:.1f}% across articles. This indicates a spread of contribution counts in aggregate, but it cannot identify who contributors are or why they edit. The cleaned `is_minor` column is `True` for all {int(summary.revision_records):,} rows; consequently every article's minor-edit percentage is 100%. This field has no variation and is not useful for comparing editing behavior until its provenance is verified.

### Event-category comparison

The category analysis compares medians and article counts because the category sizes range from two to ten articles and the article-level distributions are skewed. It is descriptive: category definitions are researcher-assigned, overlapping interpretations are possible, and several category medians are based on very small samples.

{md_table(category_summary, ['primary_category','articles','median_revisions_per_article','median_revisions_per_year','median_growth_pct'], 10)}

All categories also show a 100% median minor-edit rate because the input flag is constant. That apparent similarity is a data limitation rather than evidence that editing behavior is the same across event types.

### Significant revision events and content changes

Step 5 retrieved {int((diff_status.retrieval_status=='retrieved').sum())} of {len(diff_status)} selected parent/revision comparisons. Inspection confirms several non-substantive events. The Russian Revolution and World War I examples include large vandalism-like text insertions; the World War I page then shows one verified revert, a repeat insertion, and a second verified revert. The JFK and World War II examples also contain nonsensical or repetitive text. By contrast, the recent Vietnam War change adds maintenance templates, the Emancipation Proclamation example adds a bibliography entry, and the American Civil War example removes one casualty-list item. These diffs document edits, not the truth of the affected historical claims. Each row below preserves the revision IDs and a direct diff link for verification.

{md_table(case_studies, ['article_title','timestamp','revision_id','change_classification'], 11)}

The World War I sequence is especially informative as an editorial process example: revision 30587516 adds a large profane text block; revision 30587560 removes it about 36 seconds later; revision 30587586 adds it again about 27 seconds after that. {second_revert_note} The verified revisions document a vandalism/reversion cycle in the page history, not a change to historical documentation. See the associated links in `step05_revision_case_studies.csv`.

## Answers to the Research Questions

1. **How does coverage evolve?** The collected records show activity across a long period and substantial differences in record counts and observed page sizes. Completeness and page-creation dates are not established, so these results describe the dataset rather than all Wikipedia history.
2. **When do substantial changes occur?** Robust monthly outliers and large byte deltas identify candidate periods and revisions. They require diff inspection and independent contextual research before interpretation.
3. **Do spikes coincide with public attention?** One Arab Spring example is contemporaneous with independently documented 2011 events. No causal relationship is established; most candidate spikes still need investigation.
4. **What changes are made to historical information?** The retrieved sample includes references, maintenance templates, a list-item removal, and several vandalism/reversion events. The sample is too small to characterize all content changes or assess the historical claims themselves.
5. **Do patterns differ by event type?** The category table shows descriptive differences in medians, but small and uneven category samples, researcher-defined labels, and extreme articles limit generalization.

## Problems, Limitations, and Interpretation

The most important limitation is that this dataset is revision metadata, not the article text or full diff history. Revision count is not a direct measure of public attention, accuracy, quality, or historical importance. Size changes include references, markup, templates, and formatting. Parent-link anomalies and large size outliers need source-level validation. Contributor metadata supports only aggregate classifications. The constant minor-edit field should be traced to the cleaning or collection process. Category comparisons may change under a different mapping, and category groups contain few pages. External context is incomplete and should be linked to dated independent sources.

## Future Work

First, run the Step 5 retrieval cell where the Wikimedia API is reachable and document a small set of verified diffs across multiple articles. Second, audit the 84 non-earlier parent links, 23 parent IDs absent from the collected file, and large page-size outliers. Third, verify whether the collected revision record is complete or sampled and obtain page creation dates. Fourth, investigate the largest monthly spikes with dated external sources while separating coincidence from cause. Fifth, confirm why `is_minor` is constant and correct the source data if necessary. These actions would make a report about content evolution stronger than one based solely on activity counts.

## Team Contributions

[Add team members and verified contributions. This information was not present in the project files available for this analysis.]

## Repository

[Insert the final repository link.]
'''
(OUTPUT_DIR/'final_report_draft.md').write_text(report,encoding='utf-8')
print('Updated',OUTPUT_DIR/'final_report_draft.md','with',len(report.split()),'words')


Updated C:\Users\Jackson\Desktop\class\arch\finalProject\analysis\analysis_results\final_report_draft.md with 2231 words
